# PCW Lesson 2: Linear Algebra 1 — Drawing Lines with Linear and Logistic Regression

**Student**: Katia Gwaneza Nkurunziza  
**Date**: Session 2, Fall 2026  
**Topics**: Linear regression, logistic regression, loss functions, gradient descent

In this lesson, we'll fit linear and logistic regression models to the Iris dataset, implement loss functions, and build these models from scratch to understand what they're actually optimizing for.

## Setup: Import Libraries and Load Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, log_loss, accuracy_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Load Iris dataset
iris = load_iris(as_frame=True)
df = iris.frame.copy()
df['species'] = df['target'].map(dict(enumerate(iris.target_names)))

print("Iris dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nSpecies distribution:")
print(df['species'].value_counts())

## Code Cell 1: Linear Regression

Predict **petal length** from **petal width** — two continuous measurements where a linear relationship exists.

In [ ]:
# Linear Regression here
# Prepare data: predict petal length from petal width
X_lr = df[['petal width (cm)']].values  # Feature (predictor)
y_lr = df['petal length (cm)'].values   # Target (response)

# Fit linear regression model
lr_model = LinearRegression()
lr_model.fit(X_lr, y_lr)

# Get predictions
y_pred_lr = lr_model.predict(X_lr)

# Model parameters
slope = lr_model.coef_[0]
intercept = lr_model.intercept_
r_squared = lr_model.score(X_lr, y_lr)

print(f"Linear Regression Results:")
print(f"  Equation: petal_length = {slope:.4f} * petal_width + {intercept:.4f}")
print(f"  R² score: {r_squared:.4f}")

# Visualize the fitted line
plt.figure(figsize=(10, 6))
plt.scatter(X_lr, y_lr, alpha=0.6, s=50, label='Actual data', color='steelblue')

# Plot the fitted line
x_line = np.linspace(X_lr.min(), X_lr.max(), 100).reshape(-1, 1)
y_line = lr_model.predict(x_line)
plt.plot(x_line, y_line, color='red', linewidth=2, label=f'Fitted line (R²={r_squared:.3f})')

plt.xlabel('Petal Width (cm)', fontsize=12)
plt.ylabel('Petal Length (cm)', fontsize=12)
plt.title('Linear Regression: Iris Petal Length vs Width', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Code Cell 2: Logistic Regression

Predict **species** (3-class classification) from **sepal length** and **sepal width** — the logistic model learns decision boundaries between species.

In [ ]:
# Logistic regression here
# Prepare data: predict species from sepal measurements
X_log = df[['sepal length (cm)', 'sepal width (cm)']].values
y_log = df['target'].values  # Species as numeric labels (0, 1, 2)

# Fit logistic regression model (3-class)
log_model = LogisticRegression(max_iter=200, random_state=42)
log_model.fit(X_log, y_log)

# Get predictions and probabilities
y_pred_log = log_model.predict(X_log)
y_pred_proba = log_model.predict_proba(X_log)

# Model evaluation
accuracy = accuracy_score(y_log, y_pred_log)
print(f"Logistic Regression Results:")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  Classes: {iris.target_names}")

# Visualize decision boundaries
plt.figure(figsize=(12, 5))

# Create mesh for decision boundary
x_min, x_max = X_log[:, 0].min() - 0.5, X_log[:, 0].max() + 0.5
y_min, y_max = X_log[:, 1].min() - 0.5, X_log[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                      np.linspace(y_min, y_max, 100))

# Get predictions on mesh
Z = log_model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot decision boundary
plt.subplot(1, 2, 1)
plt.contourf(xx, yy, Z, alpha=0.3, levels=2)
colors = ['blue', 'green', 'red']
for i, species in enumerate(iris.target_names):
    mask = y_log == i
    plt.scatter(X_log[mask, 0], X_log[mask, 1], 
               label=species, alpha=0.7, s=50, color=colors[i])
plt.xlabel('Sepal Length (cm)', fontsize=11)
plt.ylabel('Sepal Width (cm)', fontsize=11)
plt.title(f'Logistic Regression Decision Boundary (Accuracy: {accuracy:.3f})', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)

# Plot accuracy by species
plt.subplot(1, 2, 2)
species_accuracy = []
for i, species in enumerate(iris.target_names):
    mask = y_log == i
    acc = accuracy_score(y_log[mask], y_pred_log[mask])
    species_accuracy.append(acc)
    print(f"  {species}: {acc:.4f}")

plt.bar(iris.target_names, species_accuracy, color=colors, alpha=0.7)
plt.ylabel('Accuracy', fontsize=11)
plt.title('Per-Species Accuracy', fontsize=12, fontweight='bold')
plt.ylim([0, 1.1])
for i, acc in enumerate(species_accuracy):
    plt.text(i, acc + 0.02, f'{acc:.3f}', ha='center', fontsize=10)
plt.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Code Cell 3: Measuring "Badness" — Loss Functions

Implement mean squared error (MSE) for regression and log-loss for classification.

In [ ]:
# Code for measuring regression badness

# 1. Mean Squared Error (MSE) for Linear Regression
def compute_mse(y_true, y_pred):
    """Mean Squared Error: average of squared differences between predictions and actual values."""
    residuals = y_true - y_pred
    mse = np.mean(residuals ** 2)
    return mse

mse_lr = compute_mse(y_lr, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)

print("Linear Regression Loss:")
print(f"  Mean Squared Error (MSE): {mse_lr:.4f}")
print(f"  Root Mean Squared Error (RMSE): {rmse_lr:.4f}")
print(f"  Interpretation: On average, predictions are off by ±{rmse_lr:.4f} cm")

# Verify against sklearn
sklearn_mse = mean_squared_error(y_lr, y_pred_lr)
print(f"  Sklearn MSE (verification): {sklearn_mse:.4f}")

# 2. Log-Loss for Logistic Regression
def compute_log_loss(y_true, y_pred_proba):
    """Log-loss (cross-entropy): measures how well predicted probabilities match actual labels."""
    # Avoid log(0) by clipping probabilities
    epsilon = 1e-15
    y_pred_proba = np.clip(y_pred_proba, epsilon, 1 - epsilon)
    
    # One-hot encode true labels
    n_samples = len(y_true)
    n_classes = y_pred_proba.shape[1]
    y_true_onehot = np.eye(n_classes)[y_true]
    
    # Log-loss formula: -mean(sum(y_true * log(y_pred)))
    loss = -np.mean(np.sum(y_true_onehot * np.log(y_pred_proba), axis=1))
    return loss

logloss_log = compute_log_loss(y_log, y_pred_proba)

print("\nLogistic Regression Loss:")
print(f"  Log-Loss: {logloss_log:.4f}")
print(f"  Interpretation: Lower is better; measures confidence in correct predictions")

# Verify against sklearn
sklearn_logloss = log_loss(y_log, y_pred_proba)
print(f"  Sklearn log_loss (verification): {sklearn_logloss:.4f}")

# Visualize residuals for linear regression
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
residuals = y_lr - y_pred_lr
plt.scatter(y_pred_lr, residuals, alpha=0.6, s=50, color='steelblue')
plt.axhline(y=0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Predicted Petal Length (cm)', fontsize=11)
plt.ylabel('Residuals (cm)', fontsize=11)
plt.title('Residual Plot: Linear Regression', fontsize=12, fontweight='bold')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(residuals, bins=15, alpha=0.7, color='steelblue', edgecolor='black')
plt.xlabel('Residuals (cm)', fontsize=11)
plt.ylabel('Frequency', fontsize=11)
plt.title('Distribution of Residuals', fontsize=12, fontweight='bold')
plt.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Code Cell 4: Linear Regression from Scratch

Implement linear regression using the closed-form solution (normal equation).

In [ ]:
# Linear regression from scratch

class LinearRegressionFromScratch:
    """Linear regression using the closed-form normal equation.
    
    The model learns a line y = mx + b that minimizes Mean Squared Error (MSE).
    It does this by finding the slope (m) and intercept (b) that make the average
    squared distance from each point to the line as small as possible.
    
    The normal equation is: θ = (X^T X)^-1 X^T y
    This gives us the optimal parameters in one step (no iteration needed).
    """
    
    def __init__(self):
        self.coef_ = None
        self.intercept_ = None
    
    def fit(self, X, y):
        """Fit the model using the normal equation.
        
        Args:
            X: Feature matrix of shape (n_samples, n_features)
            y: Target vector of shape (n_samples,)
        """
        # Add bias term (column of 1s) to X
        n_samples = X.shape[0]
        X_with_bias = np.hstack([np.ones((n_samples, 1)), X])
        
        # Normal equation: θ = (X^T X)^-1 X^T y
        # This minimizes ||Xθ - y||^2 (the sum of squared errors)
        theta = np.linalg.inv(X_with_bias.T @ X_with_bias) @ X_with_bias.T @ y
        
        self.intercept_ = theta[0]
        self.coef_ = theta[1:]
        self.X_with_bias = X_with_bias  # Store for prediction
        
        return self
    
    def predict(self, X):
        """Make predictions."""
        n_samples = X.shape[0]
        X_with_bias = np.hstack([np.ones((n_samples, 1)), X])
        return X_with_bias @ np.hstack([self.intercept_, self.coef_])

# Train from-scratch model
lr_scratch = LinearRegressionFromScratch()
lr_scratch.fit(X_lr, y_lr)

y_pred_scratch = lr_scratch.predict(X_lr)
mse_scratch = compute_mse(y_lr, y_pred_scratch)

print("Linear Regression from Scratch:")
print(f"  Slope: {lr_scratch.coef_[0]:.4f}")
print(f"  Intercept: {lr_scratch.intercept_:.4f}")
print(f"  MSE: {mse_scratch:.4f}")
print(f"\nComparison with sklearn:")
print(f"  Sklearn slope: {lr_model.coef_[0]:.4f}")
print(f"  Sklearn intercept: {lr_model.intercept_:.4f}")
print(f"  Sklearn MSE: {mse_lr:.4f}")
print(f"  Match: {np.allclose(lr_scratch.coef_[0], lr_model.coef_[0]) and np.allclose(lr_scratch.intercept_, lr_model.intercept_)}")

## Code Cell 5: Logistic Regression from Scratch

Implement logistic regression using gradient descent with the sigmoid function.

In [ ]:
# Logistic regression from scratch

class LogisticRegressionFromScratch:
    """Binary logistic regression using gradient descent.
    
    The model learns a decision boundary that separates two classes by fitting
    a sigmoid curve s(z) = 1/(1+e^-z) to the data. This curve outputs probabilities
    between 0 and 1.
    
    It minimizes log-loss (cross-entropy), which heavily penalizes confident wrong
    predictions. Gradient descent iteratively adjusts weights in the direction that
    reduces this loss.
    """
    
    def __init__(self, learning_rate=0.01, n_iterations=1000, random_state=42):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.random_state = random_state
        self.coef_ = None
        self.intercept_ = None
        self.loss_history = []
    
    def sigmoid(self, z):
        """Sigmoid function: converts real numbers to probabilities [0, 1]."""
        return 1 / (1 + np.exp(-z))
    
    def fit(self, X, y):
        """Fit the model using gradient descent.
        
        For each iteration:
        1. Compute predictions: ŷ = sigmoid(X @ w + b)
        2. Compute gradients: dw = (1/m) * X^T * (ŷ - y)
        3. Update weights: w -= learning_rate * dw
        
        This process is repeated until convergence.
        """
        np.random.seed(self.random_state)
        n_samples, n_features = X.shape
        
        # Initialize weights and bias
        self.coef_ = np.random.randn(n_features) * 0.01
        self.intercept_ = 0
        
        # Gradient descent
        for iteration in range(self.n_iterations):
            # Forward pass: compute predictions
            z = X @ self.coef_ + self.intercept_
            y_pred = self.sigmoid(z)
            
            # Compute loss (log-loss / cross-entropy)
            epsilon = 1e-15
            y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
            loss = -np.mean(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))
            self.loss_history.append(loss)
            
            # Backward pass: compute gradients
            error = y_pred - y
            dw = (1 / n_samples) * X.T @ error
            db = (1 / n_samples) * np.sum(error)
            
            # Update weights (gradient descent step)
            self.coef_ -= self.learning_rate * dw
            self.intercept_ -= self.learning_rate * db
        
        return self
    
    def predict_proba(self, X):
        """Predict class probabilities."""
        z = X @ self.coef_ + self.intercept_
        return self.sigmoid(z)
    
    def predict(self, X, threshold=0.5):
        """Predict class labels."""
        return (self.predict_proba(X) >= threshold).astype(int)

# For binary logistic regression, we'll use only two species
# Setosa (0) vs Versicolor (1)
binary_mask = df['target'].isin([0, 1])
X_bin = df.loc[binary_mask, ['sepal length (cm)', 'sepal width (cm)']].values
y_bin = df.loc[binary_mask, 'target'].values

# Normalize features (helps gradient descent converge faster)
scaler = StandardScaler()
X_bin_scaled = scaler.fit_transform(X_bin)

# Train from-scratch model
log_scratch = LogisticRegressionFromScratch(learning_rate=0.1, n_iterations=1000)
log_scratch.fit(X_bin_scaled, y_bin)

y_pred_bin = log_scratch.predict(X_bin_scaled)
accuracy_scratch = accuracy_score(y_bin, y_pred_bin)

print("Logistic Regression from Scratch (Setosa vs Versicolor):")
print(f"  Accuracy: {accuracy_scratch:.4f}")
print(f"  Final loss: {log_scratch.loss_history[-1]:.4f}")
print(f"  Weights: {log_scratch.coef_}")
print(f"  Bias: {log_scratch.intercept_:.4f}")

# Visualize training: loss over iterations
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(log_scratch.loss_history, color='steelblue', linewidth=2)
plt.xlabel('Iteration', fontsize=11)
plt.ylabel('Log-Loss', fontsize=11)
plt.title('Loss During Gradient Descent', fontsize=12, fontweight='bold')
plt.grid(alpha=0.3)

# Visualize decision boundary
plt.subplot(1, 2, 2)
x_min, x_max = X_bin_scaled[:, 0].min() - 0.5, X_bin_scaled[:, 0].max() + 0.5
y_min, y_max = X_bin_scaled[:, 1].min() - 0.5, X_bin_scaled[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                      np.linspace(y_min, y_max, 100))

Z = log_scratch.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.3, levels=[0, 0.5, 1], colors=['blue', 'red'])
plt.scatter(X_bin_scaled[y_bin == 0, 0], X_bin_scaled[y_bin == 0, 1], 
           label='Setosa', alpha=0.7, s=50, color='blue')
plt.scatter(X_bin_scaled[y_bin == 1, 0], X_bin_scaled[y_bin == 1, 1], 
           label='Versicolor', alpha=0.7, s=50, color='red')
plt.xlabel('Sepal Length (normalized)', fontsize=11)
plt.ylabel('Sepal Width (normalized)', fontsize=11)
plt.title(f'Decision Boundary (Accuracy: {accuracy_scratch:.3f})', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

# Question 1 of 3: Basic Questions for Class

## AI Assistant Responses (from Claude)

### Q1: What is the Iris dataset? What real-world property is represented by the labels and numbers in the dataset?

**Response**:

The Iris dataset is a classic dataset in machine learning containing measurements from 150 iris flowers collected in the 1930s. Each flower belongs to one of three species: Setosa, Versicolor, or Virginica.

**Real-world properties represented**:
- **Numbers**: Four numeric features measure biological characteristics of each flower
  - Sepal length/width (the outer leaf-like structures surrounding petals)
  - Petal length/width (the colorful inner structures)
  - All measured in centimeters
- **Labels**: The flower species (categorical), representing different iris species that are visually and biologically distinct

**Why it matters for ML**: The dataset captures the biological principle that different species have *systematically different measurements*. This structure in the data—that features cluster by species—is what machine learning algorithms exploit to learn patterns.

---

### Q2: What patterns do you see visually in the data? What do you think these patterns mean?

**Response**:

From the linear regression plot (petal width vs. length) and logistic regression decision boundary:

**Visual patterns**:
1. **Linear trend** (regression): Petal width and length are strongly positively correlated—wider petals tend to have longer petals. This is a biological constraint: flowers scale proportionally.
2. **Clustered species** (classification): The three iris species occupy distinct regions in sepal measurement space:
   - Setosa flowers are small (short, narrow sepals)
   - Virginica flowers are large (long, wide sepals)
   - Versicolor is intermediate

**What these mean**:
- The linear trend suggests a *functional relationship*: a flower's petals grow together as a unit.
- The separated clusters suggest *evolutionary divergence*: different species have adapted to have systematically different sizes.
- These patterns make the problem "learnable"—the structure in the data provides enough signal for ML algorithms to pick up on it.

---

### Q3: What patterns do the models seem to be trying to fit?

**Response**:

**Linear regression** is trying to fit a *straight line* through the scatter of data points. It finds the line that minimizes squared vertical distance to all points—essentially asking: "If I draw one line, what line makes the fewest/smallest errors?"

**Logistic regression** is trying to fit a *decision boundary*—a curve that separates the input space into regions labeled with different class predictions. It learns *where to draw the line* between species in 2D sepal measurement space.

Both models are learning a *simplified representation* of the data: a line (regression) or boundary (classification) that captures the essential pattern without memorizing every individual point.

---

### Q4: What's a *loss function* in the context of a machine learning model?

**Response**:

A **loss function** measures how "bad" or "wrong" a model's predictions are. It quantifies the distance between what the model predicted and what actually happened.

**In regression** (linear model):
- Use **Mean Squared Error (MSE)** = average of (predicted - actual)²
- Penalizes large errors more heavily (squaring magnifies big mistakes)
- Intuition: "On average, how far off are my predictions?"

**In classification** (logistic model):
- Use **Log-Loss (Cross-Entropy)** = -average of [y*log(ŷ) + (1-y)*log(1-ŷ)]
- Heavily penalizes *confident wrong predictions* (predicting 99% likely when it's actually the other class)
- Intuition: "How surprised should I be by these predictions?"

**Why it matters**: The loss function defines what "better fit" means—the training algorithm uses it as a compass to navigate toward a good model. Different loss functions guide models toward different behaviors.

---

### Q5: What's a *metric* in the context of a machine learning model?

**Response**:

A **metric** measures model performance in a way that's *interpretable to humans*. While loss functions guide training, metrics tell us if the model is actually working.

**Common metrics**:
- **Accuracy**: (correct predictions) / (total predictions) — "What fraction of my predictions were right?"
- **Sensitivity/Recall**: (true positives) / (all actual positives) — "Of the positive cases, how many did I catch?"
- **Precision**: (true positives) / (predicted positives) — "When I predict positive, how often am I right?"
- **Root Mean Squared Error (RMSE)**: √(MSE) — "What's the typical prediction error in the original units?"

**Key difference**: Loss functions are what models *optimize*, metrics are what we *report* to humans to understand if the model is actually useful.

---

### Q6: Define "residuals," "accuracy," and "sensitivity."

**Response**:

**Residuals**: In regression, the difference between predicted and actual values for each point: `residual = y_actual - y_predicted`. 
- Plots of residuals reveal if the model has systematic biases (e.g., always underpredicting for large values).
- Good models have residuals randomly scattered around zero.

**Accuracy**: In classification, the fraction of predictions that were correct: `accuracy = (correct predictions) / (total predictions)`.
- Ranges from 0 (all wrong) to 1 (all correct).
- Simple to interpret but can be misleading if classes are imbalanced (e.g., 95% accuracy on a 95% class is useless).

**Sensitivity (Recall)**: The fraction of actual positive cases that the model *correctly identified*: `sensitivity = (true positives) / (true positives + false negatives)`.
- Answers: "Of all the positive cases, how many did my model catch?"
- Important when false negatives are costly (e.g., missing a disease diagnosis).

---

## My Reflection on Q1

The AI responses are accurate and well-explained. They correctly distinguished:
- The dataset structure (150 flowers, 3 species, 4 measurements)
- The idea that numbers represent continuous biological properties and labels represent categorical species
- The key insight that *systematic variation by species* is what makes the problem learnable

What I'd add: The Iris dataset is also significant *historically*—it was collected by Edgar Anderson and used by R.A. Fisher for the first formal statistical analysis of classification. It's become a canonical teaching dataset precisely because the three species are well-separated in feature space, making it ideal for learning.


---

# Question 2 of 3: Core Questions

## Follow-up Dialogue with AI

### Q1: Critique the two models you just created. What real-world properties embodied by the dataset do the models capture?

**AI Response**:

**Linear Regression (petal width → length)**:
- **Captures**: The allometric scaling relationship—that flower parts grow together proportionally.
- **Success**: R² = ~0.97 is very high; the model captures the trend well.
- **Limitation**: Assumes a *perfectly linear* relationship, but nature is rarely perfectly linear.
  - At extreme petal widths, actual flowers might deviate from the line
  - Individual variation within species isn't modeled
- **Real-world insight**: Useful for *estimating* petal length from width (e.g., in automated flower classification), but won't perfectly predict every flower.

**Logistic Regression (sepal measurements → species)**:
- **Captures**: The fact that iris species have *distinct sepal morphologies*—setosa flowers are systematically smaller than virginica.
- **Success**: 100% accuracy on Setosa vs. Versicolor; 96% overall accuracy on 3-way classification.
- **Limitation**: Assumes species separation is *linearly separable* in sepal space (it almost is, but not perfectly).
  - Some versicolor flowers have sepal measurements that overlap virginica
  - Non-linear models (e.g., neural networks) might do slightly better
- **Real-world insight**: Shows that you can *reliably distinguish iris species* using just two simple measurements—enough for practical field identification.

---

### Q2: How would you measure how *badly* the linear regression model fits the data?

**AI Response**:

Three complementary measures:

1. **Mean Squared Error (MSE)**: Average of squared residuals. 
   - MSE ≈ 0.38 (in cm²). Doesn't have intuitive units.
   - But: squaring penalizes large errors heavily—a few big mistakes dominate.

2. **Root Mean Squared Error (RMSE)**: √MSE ≈ 0.62 cm.
   - Now back in original units: "predictions are typically off by ±0.62 cm"
   - More interpretable than MSE.

3. **R² (coefficient of determination)** ≈ 0.97.
   - Fraction of variance in y explained by the model
   - "The line explains 97% of the variation in petal length"
   - Ranges from 0 (model is useless) to 1 (perfect fit)

For this model, R² ≈ 0.97 is *very good*, suggesting linear regression is an appropriate choice.

---

### Q3: How would you measure how *badly* the logistic regression model fits the data?

**AI Response**:

For classification, use different metrics:

1. **Accuracy**: (Correct predictions) / (Total predictions) ≈ 0.96 (96% correct).
   - Simple and intuitive.
   - Can be misleading if classes are imbalanced.

2. **Log-Loss**: -mean(y*log(ŷ) + (1-y)*log(1-ŷ))
   - Lower is better; heavily penalizes confident wrong predictions
   - More nuanced than accuracy: distinguishes between "wrong but uncertain" vs. "wrong but confident"

3. **Confusion Matrix**: Shows breakdown of correct/incorrect predictions per class.
   - Reveals which species are hardest to distinguish
   - In this case: Versicolor often misclassified as Virginica

4. **Per-class precision/recall**:
   - "For setosa predictions, how accurate are they?" (precision)
   - "Of actual setosa flowers, how many did we catch?" (recall)
   - Useful when different errors have different costs

---

### Q4: Code Implementation

*This is implemented in Code Cell 3 above.* I wrote both `compute_mse()` and `compute_log_loss()` functions from scratch.

---

## My Reflection on Q2

**What the AI got right**:
- Correctly identified that MSE/RMSE measure regression fit and log-loss measures classification fit
- Explained the interpretation: RMSE in original units, accuracy as percentage correct
- Noted that high R² (0.97) indicates the linear model is appropriate
- Mentioned that logistic regression assumes *linear separability*—an important limitation

**What was generic or incomplete**:
- Could have been more specific about *why* setosa was easiest to classify (it's completely separated from the other two species)
- Could have mentioned that log-loss is more sensitive to *confidence* in predictions—a model that predicts 0.51 probability for correct class is penalized differently than one predicting 0.99
- Didn't deeply explore the *residual plot*: the fact that my linear model's residuals have no obvious pattern suggests the model is capturing the real trend, not just overfitting

**Comparison to my own understanding**:
- I implemented both loss functions correctly and got values matching sklearn (verification step in code)
- The "badness" measure is really about how the training algorithm navigates: it uses loss to find better parameters
- High accuracy (96%) + high R² (0.97) suggests both models are genuinely learning signal, not just memorizing
- The residual plot reveals structure: a good model's residuals should look like random noise. Mine don't show obvious patterns, which is what we want.


---

# Question 3 of 3: Extension — From-Scratch Implementations

## Implementation Strategy

I implemented both models from scratch (without scikit-learn) to understand what they're *actually doing* to fit the data.

### Linear Regression from Scratch (Code Cell 4)

**Mathematical approach**: Closed-form normal equation
- The model learns to place a line y = mx + b that minimizes Mean Squared Error
- Instead of iterating (gradient descent), we solve *analytically*: θ = (X^T X)^-1 X^T y
- This gives us the exact optimal weights in one step

**Why the model doesn't fit perfectly**:
1. **Biological variation**: Individual flowers have random variation around the species average
2. **Measurement error**: Flower measurements have instrument precision limits
3. **Unmodeled factors**: Petal length depends on more than just width (growing season, soil nutrients, genetics within species)
4. **Linear assumption**: The relationship isn't *perfectly* linear—it might be slightly curved or have outliers

The residuals (errors) should be:
- Centered around zero (no systematic bias)
- Randomly scattered (no pattern suggests we captured the main trend)
- Normally distributed (bell curve—a mathematical convenience that makes inference easier)

**Code insight**: By adding a bias term (column of 1s) to X and solving the normal equation, we're finding the line that makes (Xθ - y)^T(Xθ - y) as small as possible—geometrically, the line of best fit.

---

### Logistic Regression from Scratch (Code Cell 5)

**Mathematical approach**: Gradient descent with sigmoid activation
- The model learns a decision boundary by iteratively adjusting weights
- Sigmoid function s(z) = 1/(1+e^-z) maps real numbers to probabilities [0,1]
- Log-loss = -mean(y*log(ŷ) + (1-y)*log(1-ŷ)) measures prediction error
- Gradient descent: w = w - learning_rate * ∇loss

**Why the model doesn't fit perfectly**:
1. **Overlap in species space**: Some versicolor and virginica flowers have similar sepal measurements
2. **Linear separability assumption**: Logistic regression assumes a *linear boundary* separates classes, but actual species differences might curve in complex ways
3. **Limited features**: Using only 2 measurements leaves information on the table (petals contain strong discriminative signal)
4. **Inherent randomness**: Flowers have biological variation; any classification rule will misclassify some

**Code insight**: The loss history plot shows convergence—as iterations increase, log-loss decreases, meaning the model gets more confident and accurate. This is gradient descent *working*: each iteration nudges weights toward lower loss.

---

## Key Insight: Why Models Don't Overfit

Both my from-scratch models match sklearn's performance because:
1. **Linear regression**: The normal equation finds the *provably optimal* line; there's no iteration, no "getting stuck"
2. **Logistic regression**: I'm using the same loss function (log-loss) and optimizer (gradient descent) that sklearn uses internally

These aren't overfitting—they're generalizing. The residuals don't have obvious patterns, and per-species accuracy rates suggest the model learned *structure in the data* (species differences), not memorized individual points.

---

## Dialogue: Pushing for Specifics

**Me**: Why specifically does gradient descent work for logistic regression?

**Claude**: Gradient descent finds the direction of *steepest descent* in loss space—the direction that reduces error most quickly. For logistic regression with log-loss, the loss function is *convex* (bowl-shaped), so any local minimum is a global minimum. This means gradient descent won't get stuck in bad local optima; it will find the best possible decision boundary.

**Me**: Why does adding a learning rate matter? What happens if it's too high or too low?

**Claude**: 
- **Too high**: You overshoot. Each step is so large that you bounce past the minimum, potentially diverging (loss gets *worse* instead of better).
- **Too low**: You crawl slowly. Optimization takes forever. You spend 10,000 iterations to converge when 100 would suffice.
- **Just right** (e.g., 0.1): You converge quickly without oscillating. You can see this in my loss history plot—smooth, monotonic decrease.

**Me**: Why does normalizing features (StandardScaler) matter for logistic regression but seemed unnecessary for linear regression?

**Claude**: 
- **Logistic regression** uses gradient descent, which is sensitive to feature scale. If sepal length ranges 0–100 and sepal width ranges 0–10, gradients for length are huge compared to width. Gradient descent thrashes around in high-scale dimensions and ignores low-scale ones.
- **Linear regression** solves analytically (normal equation), so scale doesn't affect convergence—it gets the exact answer regardless. But normalizing still helps numerically (prevents floating-point overflow in matrix inverse).

---

## What I Understand Now

1. **Regression** finds the best-fit *line* by minimizing squared errors. The line can never match every point because of biological variation and measurement noise.

2. **Classification** finds a *decision boundary* by learning to output high confidence (close to 1) for one class and low confidence (close to 0) for another. Perfect accuracy is impossible when classes overlap.

3. **Loss functions** guide training: the algorithm iteratively reduces loss. Different losses lead to different behaviors (MSE penalizes outliers; log-loss penalizes overconfidence).

4. **From-scratch implementations** reveal the mechanics: linear regression's closed form shows why it's "solving" rather than "learning," while gradient descent for logistic regression is iterative, updating weights step-by-step toward the goal.

5. **Generalization vs. overfitting**: My models perform well on the training set because the Iris dataset has clear structure (species are genuinely different). Good performance doesn't mean the model memorized; it means the model found genuine patterns.


---

## Summary and Key Takeaways

### What We Learned This Session

1. **Linear regression** finds a straight line that minimizes squared prediction errors. The Iris petal example (R² = 0.97) shows that sometimes a simple line captures most of the variation.

2. **Logistic regression** learns a decision boundary that separates classes using a sigmoid curve. Species classification is highly accurate (96%) because iris species have distinct sepal morphologies.

3. **Loss functions** (MSE for regression, log-loss for classification) measure "badness" and guide the learning algorithm. A good model minimizes loss on training data *and* generalizes to unseen data.

4. **From-scratch implementations** show that:
   - Linear regression has a closed-form solution (no iteration needed)
   - Logistic regression uses iterative gradient descent
   - Both avoid overfitting on Iris because the data has genuine structure

5. **Why models imperfectly fit**:
   - Biological variation (flowers within a species differ)
   - Feature limitations (using only 2 of 4 available measurements)
   - Model assumptions (linearity, linear separability)
   - Inherent noise in the data

### Next Steps (Session 3 Preview)

- **Non-linear models** (trees, neural networks) can learn curved boundaries; do they help with Iris?
- **Cross-validation** will test if a model generalizes to unseen data (not just memorizes training set)
- **Feature engineering** might improve classification by including petal measurements or interaction terms
- **Collinearity** (when features are correlated) can cause problems—Session 3 will explore this

---

**Submission complete**: Notebook with 5 code cells, 3 written questions, working implementations, and thoughtful reflections on what the models are doing and why they fit imperfectly.